# Experiment Tracking and Lineage Lab

**Module 3 -- Tracking | Assignment Notebook**

---

## Scenario

FraudShield Risk Analytics has trained multiple XGBoost fraud-detection models
with different hyperparameter configurations. The data-science team faces a
governance challenge: which model was trained on which version of the data, with
what parameters, and where is the resulting artifact stored?

In this lab you will:

1. Create a SageMaker Experiment and run three training jobs with different
   hyperparameters under it.
2. Compare runs programmatically to identify the best-performing configuration.
3. Explore the Lineage Graph for the best model, tracing its provenance from
   endpoint to dataset.
4. Compile a reproducibility report documenting everything needed to recreate
   the model.

**Runtime environment:** SageMaker Studio on `ml.t3.medium` (Free Tier eligible).  
**Training instance:** `ml.m5.xlarge` (Free Tier eligible for training jobs).  
**Region:** us-east-1

---

*Complete each step sequentially. Cells marked with TODO require your input.*

In [ ]:
%pip install -q sagemaker boto3 pandas

import sagemaker
import boto3
import json
import time
from datetime import datetime
from sagemaker.experiments.run import Run
from sagemaker.experiments.experiment import Experiment
from sagemaker.analytics import ExperimentAnalytics
from sagemaker.xgboost import XGBoost

sm_session = sagemaker.Session()
sm_client = sm_session.sagemaker_client
region = sm_session.boto_region_name
role = sagemaker.get_execution_role()
default_bucket = sm_session.default_bucket()

PREFIX = "fraudshield-tracking"
EXPERIMENT_NAME = f"fraudshield-lab-{datetime.now().strftime('%Y%m%d-%H%M')}"

TRAIN_S3 = f"s3://{default_bucket}/{PREFIX}/processed/train.csv"
VALIDATION_S3 = f"s3://{default_bucket}/{PREFIX}/processed/test.csv"

print(f"Region          : {region}")
print(f"Role            : {role.split('/')[-1]}")
print(f"Default bucket  : {default_bucket}")
print(f"Experiment name : {EXPERIMENT_NAME}")
print(f"Train data      : {TRAIN_S3}")
print(f"Validation data : {VALIDATION_S3}")

## Concept Connection: Why Experiment Tracking?

Without a structured experiment tracking system, comparing training runs means
opening each training job individually in the console, manually recording
metrics in a spreadsheet, and hoping no one overwrites or forgets a result.
This ad-hoc approach breaks down quickly when a team runs dozens of experiments
per week.

SageMaker Experiments provides a managed solution:

- **Organized comparison:** All runs in an Experiment are visible in a single
  table with sortable columns for every parameter and metric.
- **Searchable metadata:** Custom parameters (like `data_version` or
  `feature_set`) are stored as searchable metadata, not buried in logs.
- **Reproducibility:** Every Run captures the exact hyperparameters, data paths,
  container image, and metric values -- the inputs to a reproducibility report.
- **Integration with HPO:** Hyperparameter tuning jobs automatically create one
  Run per trial, so HPO results appear in the same comparison view as manual
  runs.

For FraudShield, this means the compliance team can answer "which model
performed best on last quarter's data?" with a single query instead of a manual
investigation.

---

## Step 1: Create an Experiment and Run Training Jobs

Create a SageMaker Experiment, then launch three training jobs with different
hyperparameter configurations under it. Each job is wrapped in a `Run` context
manager, which automatically associates the job with the Experiment and logs
the parameters you specify.

The three configurations test a range of model complexity:
- **Conservative:** low depth, low learning rate -- slower learning, less
  overfitting risk
- **Balanced:** moderate depth and learning rate -- the middle ground
- **Aggressive:** high depth, high learning rate -- faster learning, higher
  overfitting risk

In [ ]:
experiment = Experiment.create(
    experiment_name=EXPERIMENT_NAME,
    description=(
        "FraudShield lab: compare XGBoost hyperparameter configurations "
        "for fraud detection and select the best model."
    ),
    sagemaker_session=sm_session,
)

print(f"Created experiment: {EXPERIMENT_NAME}")

In [ ]:
run_configs = [
    {"run_name": "conservative",  "max_depth": 3, "eta": 0.1, "num_round": 150, "scale_pos_weight": 10},
    {"run_name": "balanced",      "max_depth": 5, "eta": 0.2, "num_round": 100, "scale_pos_weight": 10},
    {"run_name": "aggressive",    "max_depth": 8, "eta": 0.3, "num_round": 80,  "scale_pos_weight": 15},
]

training_jobs = {}

for config in run_configs:
    run_name = config["run_name"]
    print(f"Starting run: {run_name} (max_depth={config['max_depth']}, eta={config['eta']})")

    with Run(
        experiment_name=EXPERIMENT_NAME,
        run_name=run_name,
        sagemaker_session=sm_session,
    ) as run:
        run.log_parameter("max_depth", config["max_depth"])
        run.log_parameter("eta", config["eta"])
        run.log_parameter("num_round", config["num_round"])
        run.log_parameter("scale_pos_weight", config["scale_pos_weight"])
        run.log_parameter("objective", "binary:logistic")
        run.log_parameter("eval_metric", "auc")
        run.log_parameter("data_version", "2026-Q1")

        xgb = XGBoost(
            entry_point="train.py",
            role=role,
            instance_count=1,
            instance_type="ml.m5.xlarge",
            framework_version="1.5-1",
            hyperparameters={
                "max_depth": config["max_depth"],
                "eta": config["eta"],
                "objective": "binary:logistic",
                "num_round": config["num_round"],
                "eval_metric": "auc",
                "scale_pos_weight": config["scale_pos_weight"],
            },
        )

        xgb.fit(
            inputs={
                "train": TRAIN_S3,
                "validation": VALIDATION_S3,
            }
        )

        training_jobs[run_name] = xgb.latest_training_job.name
        print(f"  Completed: {xgb.latest_training_job.name}")

print()
print("All training jobs:")
for name, job in training_jobs.items():
    print(f"  {name}: {job}")

### Presentation Checkpoint

Navigate to **SageMaker Studio > Experiments** in the console. Open your
experiment and verify that all three runs appear in the list.

**Demonstrate:**
1. Select all three runs and click **Analyze** to open the comparison chart.
2. Sort by the `validation:auc` metric column to identify the best run.
3. Create a scatter plot with `max_depth` on the X-axis and `validation:auc`
   on the Y-axis to visualize the relationship between tree depth and model
   performance.

---

## Step 2: Compare Runs Programmatically

Use `ExperimentAnalytics` to retrieve all runs as a pandas DataFrame, then
identify the best-performing run by the target metric. This programmatic
approach is essential when you have many runs or need to automate model
selection in a pipeline.

In [ ]:
analytics = ExperimentAnalytics(
    experiment_name=EXPERIMENT_NAME,
    sagemaker_session=sm_session,
)

runs_df = analytics.dataframe()

print(f"Total runs: {len(runs_df)}")
print()
print("Available columns:")
for col in sorted(runs_df.columns):
    print(f"  {col}")

print()
print("Run comparison:")
display_cols = [
    col for col in runs_df.columns
    if any(k in col.lower() for k in ["run", "trial", "max_depth", "eta", "auc", "status"])
]
if display_cols:
    print(runs_df[display_cols].to_string(index=False))
else:
    print(runs_df.head())

In [ ]:
auc_cols = [col for col in runs_df.columns if "auc" in col.lower()]

best_training_job = None

if auc_cols:
    metric_col = auc_cols[0]
    best_idx = runs_df[metric_col].idxmax()
    best_run = runs_df.loc[best_idx]
    print(f"Best run by {metric_col}:")
    print(f"  Run: {best_run.get('RunName', best_run.get('TrialComponentName', 'unknown'))}")
    print(f"  {metric_col}: {best_run[metric_col]}")
else:
    print("No AUC metric column found. Inspect the columns above to identify the metric.")

if training_jobs:
    best_training_job = list(training_jobs.values())[1]
    print(f"\nUsing training job for lineage exploration: {best_training_job}")

## Concept Connection: Lineage Entities

Lineage Tracking records the full provenance chain for every model. SageMaker
automatically creates four types of lineage entities:

| Entity | Represents | FraudShield Example |
|---|---|---|
| **Artifact** | Data objects and model files | S3 URI for train.csv, model.tar.gz, container image |
| **Action** | Processing steps | The training job that produced a model |
| **Context** | Grouping containers | The Experiment, a Pipeline execution, an Endpoint |
| **Association** | Directed relationships | "Training Job used this Dataset" (ContributedTo) |

By querying the lineage graph, you can answer compliance questions: "Which data
was used to train the model currently in production?" and impact analysis
questions: "If dataset X has a quality issue, which models are affected?"

The lineage graph is built automatically. Every Training Job, Processing Job,
and endpoint deployment creates lineage entities and associations without any
additional code.

---

## Step 3: Explore Lineage

Query the lineage graph for the best model. Trace upstream to identify every
data source, processing step, and container image that contributed to the model.
This is the same query a compliance auditor would run to verify model
provenance.

In [ ]:
from sagemaker.lineage.artifact import Artifact
from sagemaker.lineage.query import LineageQuery, LineageFilter, LineageEntityEnum

model_artifact = None
upstream = None

if best_training_job is None:
    print("No training job available for lineage query.")
else:
    job_info = sm_client.describe_training_job(TrainingJobName=best_training_job)
    model_s3 = job_info["ModelArtifacts"]["S3ModelArtifacts"]

    print(f"Training job  : {best_training_job}")
    print(f"Model artifact: {model_s3}")
    print()

    model_artifacts_iter = Artifact.list(
        source_uri=model_s3,
        sagemaker_session=sm_session,
    )
    for art in model_artifacts_iter:
        model_artifact = art
        break

    if model_artifact is None:
        print("Lineage artifact not yet available. SageMaker creates lineage")
        print("entities asynchronously. Wait a few minutes and re-run this cell.")
    else:
        print(f"Model artifact ARN: {model_artifact.artifact_arn}")
        print()

        query = LineageQuery(sm_session)
        upstream = query.query(
            start_arns=[model_artifact.artifact_arn],
            direction="Ascendants",
            include_edges=True,
        )

        print("Provenance chain (upstream from model):")
        print()
        for vertex in upstream.vertices:
            short = vertex.arn.split("/")[-1]
            print(f"  [{vertex.lineage_entity_type}] {short}")

        print()
        print("Associations:")
        for edge in upstream.edges:
            src = edge.source_arn.split("/")[-1]
            dst = edge.destination_arn.split("/")[-1]
            print(f"  {src} --{edge.association_type}--> {dst}")

---

## Step 4: Build a Reproducibility Report

Combine Experiment metadata (hyperparameters, metrics) with Lineage data (data
sources, container image, provenance chain) into a single reproducibility
report. This report captures everything needed to recreate the model:

- **Data:** Exact S3 URIs and versions of training and validation datasets
- **Code:** Container image URI (pinned to a specific version)
- **Configuration:** All hyperparameters
- **Environment:** Instance type, framework version
- **Lineage:** Full upstream provenance chain

For full reproducibility, you would also need: random seed (set in the training
script), exact library versions within the container, and data shuffle order.

In [ ]:
if best_training_job is None:
    print("No training job available to build report.")
else:
    job_info = sm_client.describe_training_job(TrainingJobName=best_training_job)

    report = {
        "report_title": "FraudShield Model Reproducibility Report",
        "generated_at": datetime.now().isoformat(),
        "model_identity": {
            "training_job": best_training_job,
            "model_artifact_s3": job_info["ModelArtifacts"]["S3ModelArtifacts"],
            "creation_date": str(job_info["CreationTime"]),
            "experiment": EXPERIMENT_NAME,
        },
        "training_configuration": {
            "algorithm_image": job_info["AlgorithmSpecification"]["TrainingImage"],
            "instance_type": job_info["ResourceConfig"]["InstanceType"],
            "instance_count": job_info["ResourceConfig"]["InstanceCount"],
            "hyperparameters": job_info["HyperParameters"],
        },
        "data_provenance": {
            "channels": {
                ch["ChannelName"]: ch["DataSource"]["S3DataSource"]["S3Uri"]
                for ch in job_info["InputDataConfig"]
            },
            "recommendation": (
                "Enable S3 versioning and record the Version ID, or use "
                "Feature Store with event timestamps for point-in-time reproducibility."
            ),
        },
        "environment": {
            "container_image": job_info["AlgorithmSpecification"]["TrainingImage"],
            "region": region,
        },
        "results": {
            "final_metrics": [
                {"name": m["MetricName"], "value": m["Value"]}
                for m in job_info.get("FinalMetricDataList", [])
            ],
            "training_duration_seconds": job_info.get("TrainingTimeInSeconds"),
            "billable_seconds": job_info.get("BillableTimeInSeconds"),
        },
    }

    if model_artifact is not None and upstream is not None:
        report["lineage_chain"] = {
            "upstream_entities": [
                {"arn": v.arn, "type": v.lineage_entity_type}
                for v in upstream.vertices
            ],
            "associations": [
                {
                    "source": e.source_arn,
                    "destination": e.destination_arn,
                    "type": e.association_type,
                }
                for e in upstream.edges
            ],
        }

    print("FraudShield Reproducibility Report")
    print("=" * 70)
    print(json.dumps(report, indent=2, default=str))

### Presentation Checkpoint

Present your reproducibility report. Walk through each section and explain:

1. **Model identity:** How would you use this information to find the model in
   the console?
2. **Training configuration:** If you needed to retrain this exact model, what
   would you set?
3. **Data provenance:** How would you verify that the training data has not
   changed since this model was trained?
4. **Lineage chain:** Trace the provenance path from model to raw data. Name
   each entity type you encounter (Artifact, Action, Context, Association).
5. **What is missing?** Identify at least two things not captured in this report
   that would be needed for full reproducibility (e.g., random seed, library
   versions within the container, data shuffle order).

---

## Cleanup

Delete the Experiment and all its Runs. Experiments are metadata only (no
compute cost), but cleanup keeps the console organized and is required for
this lab.

Verify that no training jobs are still running, and shut down all notebook
kernels when finished.

In [ ]:
try:
    exp = Experiment.load(
        experiment_name=EXPERIMENT_NAME,
        sagemaker_session=sm_session,
    )
    exp._delete_all(action="--force")
    print(f"Deleted experiment and all runs: {EXPERIMENT_NAME}")
except Exception as e:
    print(f"Cleanup note: {e}")
    print("The experiment may have already been deleted.")

In [ ]:
running_jobs = sm_client.list_training_jobs(
    StatusEquals="InProgress",
    MaxResults=10,
)

if running_jobs["TrainingJobSummaries"]:
    print("WARNING: The following training jobs are still running:")
    for job in running_jobs["TrainingJobSummaries"]:
        print(f"  {job['TrainingJobName']} -- {job['TrainingJobStatus']}")
    print()
    print("Stop them manually if they are from this lab.")
else:
    print("No running training jobs found.")

print()
print("Cleanup checklist:")
print("  [x] Experiment and runs deleted")
print("  [x] Verified no running training jobs")
print("  [ ] Shut down all notebook kernels (Studio > Running Terminals and Kernels)")
print("  [ ] Confirm no active endpoints (none were deployed in this lab)")

## Submission Checklist

Before submitting this lab, verify:

- [ ] Experiment created with three training runs (conservative, balanced,
      aggressive).
- [ ] Runs compared programmatically using `ExperimentAnalytics` -- you can
      identify the best-performing hyperparameter configuration.
- [ ] Lineage queried for the best model -- you can trace the provenance chain
      upstream to the training data and container image.
- [ ] Reproducibility report compiled with model identity, training
      configuration, data provenance, environment, results, and lineage chain.
- [ ] You can explain what additional information (random seed, library
      versions, data shuffle order) would be needed for full reproducibility.
- [ ] Experiment and all runs deleted.
- [ ] No running training jobs remain.
- [ ] All notebook kernels shut down.